# SCANIA Dataset Downloader

This notebook downloads and extracts the **SCANIA Component X Dataset** from the official repository at [researchdata.se](https://researchdata.se/en/catalogue/dataset/2024-34/2).

**Run this notebook once before running `Code.ipynb` or `DL_Code.ipynb`.**

The dataset will be extracted to `SCANIA Dataset/data/`, which is the path expected by both notebooks.

In [2]:
import os
import zipfile
import urllib.request
import shutil
import sys
import time

In [3]:
# --- Configuration ---

DATASET_URL   = "https://api.researchdata.se/dataset/2024-34/3/file/zip"
ZIP_PATH      = "SCANIA Dataset.zip"
EXTRACT_DIR   = "SCANIA Dataset"
DATA_DIR      = os.path.join(EXTRACT_DIR, "data")
DOCS_DIR      = os.path.join(EXTRACT_DIR, "documentation")

REQUIRED_FILES = [
    "train_operational_readouts.csv",
    "train_specifications.csv",
    "train_tte.csv",
    "test_operational_readouts.csv",
    "test_specifications.csv",
    "test_labels.csv",
    "validation_operational_readouts.csv",
    "validation_specifications.csv",
    "validation_labels.csv",
]

In [4]:
# --- Check if the dataset is already in place ---

def dataset_complete():
    """Return True if every required CSV file already exists in DATA_DIR."""
    if not os.path.isdir(DATA_DIR):
        return False
    return all(os.path.isfile(os.path.join(DATA_DIR, f)) for f in REQUIRED_FILES)

if dataset_complete():
    print("Dataset already present — nothing to do.")
    #print(f"Location: {os.path.abspath(DATA_DIR)}") #Uncomment this line to print the location of the dataset
    for f in REQUIRED_FILES:
        size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1024**2
        print(f"{f:50s} ({size_mb:>8.1f} MB)")
else:
    print("Dataset not found (or incomplete). Proceeding with download and extraction.")

Dataset not found (or incomplete). Proceeding with download and extraction.


In [5]:
# --- Download the zip archive (skipped if already complete) ---

if not dataset_complete():

    def _progress(block_count, block_size, total_size):
        """Simple progress reporter for urllib.request.urlretrieve."""
        downloaded = block_count * block_size
        if total_size > 0:
            pct = min(downloaded / total_size * 100, 100)
            bar_len = 40
            filled = int(bar_len * pct / 100)
            bar = '█' * filled + '░' * (bar_len - filled)
            downloaded_gb = downloaded / 1024**3
            total_gb     = total_size  / 1024**3
            print(f"\r  [{bar}] {pct:5.1f}%  {downloaded_gb:.2f}/{total_gb:.2f} GB",
                  end='', flush=True)
        else:
            print(f"\r  Downloaded {downloaded / 1024**2:.1f} MB ...", end='', flush=True)

    if os.path.isfile(ZIP_PATH):
        print(f"Zip archive already exists at '{ZIP_PATH}' — skipping download.")
    else:
        print(f"Downloading dataset from:\n  {DATASET_URL}")
        print("This is ~1.5 GB and may take several minutes depending on your connection.\n")
        t0 = time.time()
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH, reporthook=_progress)
        elapsed = time.time() - t0
        size_gb = os.path.getsize(ZIP_PATH) / 1024**3
        print(f"\n\nDownload complete: {size_gb:.2f} GB in {elapsed/60:.1f} min")

  https://api.researchdata.se/dataset/2024-34/3/file/zip
This is ~1.5 GB and may take several minutes depending on your connection.

  Downloaded 1578.5 MB ...

Download complete: 1.54 GB in 29.3 min


In [6]:
# --- Extract the zip archive ---

if not dataset_complete():

    print(f"Extracting '{ZIP_PATH}' …\n")

    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(DOCS_DIR, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        all_entries = zf.namelist()

        # Detect the top-level prefix inside the zip (e.g. "2024-34-2/" or "2024-34-3/")
        prefix = ""
        for entry in all_entries:
            parts = entry.split("/")
            if len(parts) >= 2 and parts[1] in ("data", "documentation"):
                prefix = parts[0] + "/"
                break

        print(f"  Detected zip prefix: '{prefix}'")
        extracted = 0

        for entry in all_entries:
            # Map  <prefix>data/<file>          → SCANIA Dataset/data/<file>
            # Map  <prefix>documentation/<file> → SCANIA Dataset/documentation/<file>
            if entry.startswith(prefix + "data/") and not entry.endswith("/"):
                filename   = os.path.basename(entry)
                dest_path  = os.path.join(DATA_DIR, filename)
                dest_dir   = DATA_DIR
            elif entry.startswith(prefix + "documentation/") and not entry.endswith("/"):
                filename   = os.path.basename(entry)
                dest_path  = os.path.join(DOCS_DIR, filename)
                dest_dir   = DOCS_DIR
            else:
                continue

            print(f"  Extracting  {entry}  →  {dest_path}")
            with zf.open(entry) as src, open(dest_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            extracted += 1

    print(f"\nExtraction complete: {extracted} files written.")

Extracting 'SCANIA Dataset.zip' …

  Detected zip prefix: '2024-34-3/'
  Extracting  2024-34-3/data/test_labels.csv  →  SCANIA Dataset/data/test_labels.csv
  Extracting  2024-34-3/data/test_operational_readouts.csv  →  SCANIA Dataset/data/test_operational_readouts.csv
  Extracting  2024-34-3/data/test_specifications.csv  →  SCANIA Dataset/data/test_specifications.csv
  Extracting  2024-34-3/data/train_operational_readouts.csv  →  SCANIA Dataset/data/train_operational_readouts.csv
  Extracting  2024-34-3/data/train_specifications.csv  →  SCANIA Dataset/data/train_specifications.csv
  Extracting  2024-34-3/data/train_tte.csv  →  SCANIA Dataset/data/train_tte.csv
  Extracting  2024-34-3/data/validation_labels.csv  →  SCANIA Dataset/data/validation_labels.csv
  Extracting  2024-34-3/data/validation_operational_readouts.csv  →  SCANIA Dataset/data/validation_operational_readouts.csv
  Extracting  2024-34-3/data/validation_specifications.csv  →  SCANIA Dataset/data/validation_specifications.

In [7]:
# --- Verify the extraction ---

print("Verifying dataset files …\n")
all_ok = True

for fname in REQUIRED_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"✓ {fname:50s} ({size_mb:>8.1f} MB)")
    else:
        print(f"✗ MISSING: {fname}")
        all_ok = False

print()
if all_ok:
    print("All dataset files are present. You can now run Code.ipynb and DL_Code.ipynb.")
else:
    print("WARNING: Some files are missing. Re-run this notebook or check the download.")

Verifying dataset files …

✓ train_operational_readouts.csv                     (  1162.7 MB)
✓ train_specifications.csv                           (     1.0 MB)
✓ train_tte.csv                                      (     0.3 MB)
✓ test_operational_readouts.csv                      (   204.9 MB)
✓ test_specifications.csv                            (     0.2 MB)
✓ test_labels.csv                                    (     0.0 MB)
✓ validation_operational_readouts.csv                (   205.6 MB)
✓ validation_specifications.csv                      (     0.2 MB)
✓ validation_labels.csv                              (     0.0 MB)

All dataset files are present. You can now run Code.ipynb and DL_Code.ipynb.
